# 06 · Sensitivity: do the results depend on our numbers?

Every number in notebook 02's configuration is our choice; the studies give directions, not magnitudes. This notebook
changes the most influential ones by about ±50% and **reruns the actual pipeline** (notebooks 02 → 05, unchanged code)
on each variant, then compares the outcomes with the base run:

- does the data still follow the studies' directions?
- does the onboarding tree still beat one preset for everyone, and by how much?
- does it still ask about the same things (first question, questions used)?
- do the presets stay similar (k = 8, matched one-to-one with the base presets)?

How it works: for each variant, notebooks 02–05 are executed with three edits made on the fly: the data folder points
to `data/sensitivity/<variant>/`, notebook 02's configuration gets the variant's change, and the cross-validation grids
in 03 and 05 are trimmed so a variant runs in a few minutes. Failed checks inside a notebook are recorded, not fatal.
Large files are deleted once a variant's measures are taken.

In [1]:
import json, re, shutil, time
from pathlib import Path
import numpy as np, pandas as pd
import nbformat
from nbclient import NotebookClient
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
NOTEBOOKS = ROOT / "notebooks"
OUT = DATA / "sensitivity"
items = pd.read_csv(DATA / "items.csv")
real = items.in_corpus.to_numpy()
ORDINAL = np.array([0.0, 1.0, 0.5, 0.5, 0.5])

## Variants

Each is a small piece of code applied to notebook 02's `CONFIG` right after it is defined.

In [2]:
VARIANTS = {
    "purpose spread x0.5": 'CONFIG["purpose_spread"] = {k: v * 0.5 for k, v in CONFIG["purpose_spread"].items()}',
    "purpose spread x1.5": 'CONFIG["purpose_spread"] = {k: v * 1.5 for k, v in CONFIG["purpose_spread"].items()}',
    "per-cell noise x0.5": 'CONFIG["cell_noise_sd"] *= 0.5; CONFIG["item_noise_sd"] *= 0.5',
    "per-cell noise x1.5": 'CONFIG["cell_noise_sd"] *= 1.5; CONFIG["item_noise_sd"] *= 1.5',
    "concern weight x0.5": 'CONFIG["concern_scale"] *= 0.5',
    "concern weight x1.5": 'CONFIG["concern_scale"] *= 1.5',
    "third-party aversion x0.5": 'CONFIG["recipient"]["ppd:thirdPartyContext"] = (0.5, "third_party_aversion", 0.35)',
    "third-party aversion x1.5": 'CONFIG["recipient"]["ppd:thirdPartyContext"] = (0.5, "third_party_aversion", 1.05)',
    "answer noise x1.5": 'CONFIG["answer_noise_sd"] *= 1.5; CONFIG["concrete_answer_slip"] *= 1.5',
}
EDITS = {   # notebook -> list of (text to find, replacement); the data folder edit is added per variant
    "03_clustering.ipynb": [("KS = [1, 2, 4, 6, 8, 12, 16, 20, 24, 32, 40]", "KS = [1, 8, 12, 40]")],
    "05_onboarding_tree.ipynb": [("DEPTHS = [2, 3, 4, 5, 6, 7]", "DEPTHS = [4, 5]")],
}

## Running a variant

In [3]:
def run_notebook(name, data_dir, config_change=None):
    nb = nbformat.read(NOTEBOOKS / name, as_version=4)
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        cell.source = cell.source.replace('DATA = ROOT / "data"', f'DATA = Path(r"{data_dir}")')
        for find, replace in EDITS.get(name, []):
            cell.source = cell.source.replace(find, replace)
    if config_change:
        at = next(i for i, c in enumerate(nb.cells) if c.cell_type == "code" and "CONFIG = {" in c.source)
        nb.cells.insert(at + 1, nbformat.v4.new_code_cell(config_change))
    NotebookClient(nb, timeout=3600, kernel_name="python3", allow_errors=True,
                   resources={"metadata": {"path": str(NOTEBOOKS)}}).execute()
    errors = []
    for c in nb.cells:
        for o in c.get("outputs", []) if c.cell_type == "code" else []:
            if o.output_type == "error":
                failed = re.findall(r"'([^']+)': np\.False_", o.evalue)   # names of the checks that did not hold
                errors += failed or [f"{o.ename}: {o.evalue[:120]}"]
    return errors

def measures(data_dir):
    d = Path(data_dir)
    D = np.load(d / "decisions.npy")
    V = ORDINAL[D]
    m = V.mean(axis=0)
    by = lambda f: pd.Series(m).groupby(items[f]).mean()
    tree = json.loads((d / "onboarding_tree.json").read_text())
    def walk(n): return [] if "preset" in n else [n["question"]] + [q for b in n["branches"] for q in walk(b["next"])]
    return {
        "allow share": float((D == 1).mean()), "deny share": float((D == 0).mean()),
        "largest pattern (PC1)": float(PCA(1, random_state=0).fit(V[::5]).explained_variance_ratio_[0]),
        "video least accepted": by("data_type").idxmin() == "ppd:contentData",
        "ads least accepted": by("purpose").idxmin() == "ppd:advertisingAndMarketing",
        "third party below maker": by("handling_context")["ppd:thirdPartyContext"] < by("handling_context")["ppd:vendorContext"],
        "tree k": tree["k"], "tree depth": tree["depth"], "questions (avg)": tree["questions_avg"],
        "tree match (real)": tree["match_real_cv"], "one preset (real)": tree["one_preset_real_cv"],
        "gain (points)": 100 * (tree["match_real_cv"] - tree["one_preset_real_cv"]),
        "first question": tree["root"]["question"], "questions used": sorted(set(walk(tree["root"]))),
    }

def preset_similarity(data_dir, k=8):
    # Match the variant's k presets one-to-one with the base presets (by per-item agreement); mean agreement of the pairs.
    base = np.load(DATA / "presets" / f"preset_items_k{k}.npy")
    var = np.load(Path(data_dir) / "presets" / f"preset_items_k{k}.npy")
    agree = np.array([[(b == v).mean() for v in var] for b in base])
    r, c = linear_sum_assignment(-agree)
    return float(agree[r, c].mean())

## Run all variants (a few minutes each)

In [4]:
results, failures = {"base": {**measures(DATA), "preset similarity to base (k=8)": 1.0}}, {}
for name, change in VARIANTS.items():
    started = time.time()
    d = OUT / name.replace(" ", "_").replace(".", "")
    shutil.rmtree(d, ignore_errors=True); (d / "corpus").mkdir(parents=True)
    shutil.copy(DATA / "items.csv", d / "items.csv")
    errs = []
    for nb_name in ["02_generate_households.ipynb", "03_clustering.ipynb", "04_presets.ipynb", "05_onboarding_tree.ipynb"]:
        errs += [f"{nb_name[:2]}: {e}" for e in run_notebook(nb_name, d, change if nb_name.startswith("02") else None)]
    results[name] = {**measures(d), "preset similarity to base (k=8)": preset_similarity(d)}
    failures[name] = errs
    for big in ["decisions.npy", "decisions_cells.npy"]:
        (d / big).unlink(missing_ok=True)
    print(f"{name}: {time.time() - started:.0f} s, {len(errs)} failed checks")
R = pd.DataFrame(results).T

purpose spread x0.5: 50 s, 2 failed checks


purpose spread x1.5: 52 s, 0 failed checks


per-cell noise x0.5: 54 s, 0 failed checks


per-cell noise x1.5: 54 s, 0 failed checks


concern weight x0.5: 57 s, 1 failed checks


concern weight x1.5: 54 s, 0 failed checks


third-party aversion x0.5: 51 s, 0 failed checks


third-party aversion x1.5: 54 s, 1 failed checks


answer noise x1.5: 54 s, 0 failed checks


## Results

In [5]:
R[["allow share", "deny share", "largest pattern (PC1)", "video least accepted", "ads least accepted", "third party below maker"]].round(3)

,allow share,deny share,largest pattern (PC1),video least accepted,ads least accepted,third party below maker
base,0.56671,0.260814,0.178831,True,True,True
purpose spread x0.5,0.570831,0.248668,0.216797,True,True,True
purpose spread x1.5,0.56194,0.275955,0.153125,True,True,True
per-cell noise x0.5,0.569147,0.255616,0.199348,True,True,True
per-cell noise x1.5,0.562727,0.269277,0.152435,True,True,True
concern weight x0.5,0.620298,0.212677,0.143579,True,True,True
concern weight x1.5,0.52397,0.302583,0.208483,True,True,True
third-party aversion x0.5,0.589673,0.246718,0.174944,True,True,True
third-party aversion x1.5,0.548613,0.273151,0.179348,True,True,True
answer noise x1.5,0.56671,0.260814,0.178831,True,True,True


In [6]:
R[["tree k", "tree depth", "questions (avg)", "tree match (real)", "one preset (real)", "gain (points)", "preset similarity to base (k=8)", "first question"]].round(3)

,tree k,tree depth,questions (avg),tree match (real),one preset (real),gain (points),preset similarity to base (k=8),first question
base,12,4,3.86,0.855,0.828,2.7,1.0,real_profile_ads
purpose spread x0.5,4,4,3.74,0.847,0.837,1.0,0.869001,concern
purpose spread x1.5,8,5,5.0,0.857,0.819,3.8,0.915597,real_profile_ads
per-cell noise x0.5,8,4,4.0,0.858,0.833,2.5,0.913365,real_profile_ads
per-cell noise x1.5,12,4,4.0,0.842,0.822,2.0,0.941406,real_profile_ads
concern weight x0.5,8,4,4.0,0.87,0.844,2.6,0.893694,real_profile_ads
concern weight x1.5,16,4,4.0,0.844,0.815,2.9,0.92341,real_profile_ads
third-party aversion x0.5,8,4,4.0,0.856,0.821,3.5,0.90332,real_viewing_advertisers
third-party aversion x1.5,20,4,3.75,0.856,0.833,2.3,0.894252,real_profile_ads
answer noise x1.5,12,4,3.91,0.849,0.828,2.1,1.0,real_profile_ads


In [7]:
base_q = set(R.loc["base", "questions used"])
pd.DataFrame({
    "questions used": R["questions used"].map(len),
    "shared with base": R["questions used"].map(lambda q: len(base_q & set(q))),
    "concrete questions used": R["questions used"].map(lambda q: sum(x.startswith("real_") for x in q)),
})

,questions used,shared with base,concrete questions used
base,6,6,4
purpose spread x0.5,7,5,4
purpose spread x1.5,10,6,4
per-cell noise x0.5,7,5,5
per-cell noise x1.5,8,5,4
concern weight x0.5,6,5,3
concern weight x1.5,6,5,4
third-party aversion x0.5,6,6,4
third-party aversion x1.5,7,6,5
answer noise x1.5,5,5,4


In [8]:
pd.Series({k: "; ".join(v) if v else "none" for k, v in failures.items()}, name="checks that failed inside the notebooks")

purpose spread x0.5          02: some households accept ads more than analy...
purpose spread x1.5                                                       none
per-cell noise x0.5                                                       none
per-cell noise x1.5                                                       none
concern weight x0.5          05: it beats one preset for everyone on all items
concern weight x1.5                                                       none
third-party aversion x0.5                                                 none
third-party aversion x1.5    05: it beats one preset for everyone on all items
answer noise x1.5                                                         none
Name: checks that failed inside the notebooks, dtype: str

## Checks

In [9]:
checks = {
    "every variant still follows the studies' directions": R[["video least accepted", "ads least accepted", "third party below maker"]].all().all(),
    "the tree beats one preset for everyone in every variant": (R["gain (points)"] > 0).all(),
    "concrete questions stay among the questions used": (R["questions used"].map(lambda q: any(x.startswith("real_") for x in q))).all(),
}
pd.Series(checks, name="holds")

every variant still follows the studies' directions        True
the tree beats one preset for everyone in every variant    True
concrete questions stay among the questions used           True
Name: holds, dtype: bool

## How to read this

- **Directions** are built into the generator, so they should hold in every variant; this is a sanity check.
- **Gain over one preset** is the main outcome. If it stays positive across variants, the conclusion "a short
  questionnaire beats one preset for everyone" does not depend on our exact numbers. How *large* the gain is does
  depend on them, mostly on how much households differ by purpose (purpose spread) and on item-level noise.
- **Preset similarity** (k = 8, best one-to-one pairing with the base presets) shows whether the presets themselves
  are stable. Low similarity means the presets mostly reflect our assumptions, which argues for calibrating with real
  data (the LLM-respondent idea or a small survey) before presets reach households.
- **Questions used** show whether onboarding should ask the same things. Concrete real-device questions are expected
  to stay useful because they read decisions directly.